# 1. Realización de Autoencoder para Predicción de Objetos a partir de videos e imagenes.


### Importación de repositorio GitHub

In [ ]:
# Reemplaza con la URL de tu repositorio de GitHub
!git clone https://github.com/Gerardo-cursos/objetos_salon

# Ahora puedes agregar código aquí para cargar las imágenes desde el repositorio clonado.
# Por ejemplo, si tus imágenes están en una carpeta llamada 'imagenes' dentro del repositorio:
# import os
# ruta_imagenes = 'tu_repositorio/imagenes'
# archivos_imagenes = [os.path.join(ruta_imagenes, f) for f in os.listdir(ruta_imagenes) if f.endswith('.jpg') or f.endswith('.png')]
# # Luego puedes usar librerías como OpenCV o Pillow para cargar y procesar las imágenes.

### Importacion de todas las Imagenes

In [ ]:
import os
import cv2
import numpy as np

ruta_principal_imagenes = '/content/objetos_salon/processed/'
lista_imagenes = []
lista_etiquetas = [] # Si necesitas etiquetas, puedes extraerlas del nombre de la subcarpeta

# Definir el tamaño deseado
ancho_deseado = 64
alto_deseado = 64

# Recorrer las subcarpetas y cargar las imágenes
for subcarpeta in os.listdir(ruta_principal_imagenes):
    ruta_subcarpeta = os.path.join(ruta_principal_imagenes, subcarpeta)
    if os.path.isdir(ruta_subcarpeta):
        print(f"Cargando imágenes de la subcarpeta: {subcarpeta}")
        for archivo_imagen in os.listdir(ruta_subcarpeta):
            if archivo_imagen.endswith('.jpg') or archivo_imagen.endswith('.png'):
                ruta_completa_imagen = os.path.join(ruta_subcarpeta, archivo_imagen)
                imagen = cv2.imread(ruta_completa_imagen, cv2.IMREAD_GRAYSCALE) # Cargar en escala de grises para autoencoder simple
                if imagen is not None:
                    # Redimensionar las imágenes
                    imagen_redimensionada = cv2.resize(imagen, (ancho_deseado, alto_deseado))
                    lista_imagenes.append(imagen_redimensionada)
                    lista_etiquetas.append(subcarpeta) # Asignar la subcarpeta como etiqueta

# Convertir la lista de imágenes a un array numpy
imagenes_cargadas = np.array(lista_imagenes)

# Normalizar los valores de píxeles a un rango entre 0 y 1
imagenes_cargadas = imagenes_cargadas.astype('float32') / 255.

print(f"Se cargaron {len(imagenes_cargadas)} imágenes.")
print(f"Dimensiones del array de imágenes: {imagenes_cargadas.shape}")

Una vez que las imágenes estén cargadas y preprocesadas, el siguiente paso sería preparar los datos para el entrenamiento del autoencoder (dividir en conjuntos de entrenamiento y prueba, etc.) y luego definir y entrenar el modelo.

### División de todos los datos en entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir los datos en conjuntos de entrenamiento y prueba
x_train, x_test, y_train, y_test = train_test_split(
    imagenes_cargadas, lista_etiquetas, test_size=0.2, random_state=42
)

print(f"Dimensiones del conjunto de entrenamiento: {x_train.shape}")
print(f"Dimensiones del conjunto de prueba: {x_test.shape}")
print(f"Número de etiquetas de entrenamiento: {len(y_train)}")
print(f"Número de etiquetas de prueba: {len(y_test)}")

## Entrenar el autoencoder

Se entrena autoencoder similar al de los ejemplos anteriores para aprender una representación latente de las imágenes de los objetos del salón.


### Autoencoder: dimension 64

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers, losses
import numpy as np

class AutoencoderObjetos(Model):
  def __init__(self):
    super(AutoencoderObjetos, self).__init__()
    # Previous architecture that resulted in a flattened latent space of 64
    self.encoder = tf.keras.Sequential([
      layers.Input(shape=(64, 64, 1)),
      layers.Conv2D(16, (3, 3), activation='relu', padding='same', strides=2), # 32x32x16
      layers.Conv2D(8, (3, 3), activation='relu', padding='same', strides=2),  # 16x16x8
      layers.Conv2D(1, (3, 3), activation='relu', padding='same', strides=2)])  # 8x8x1 = 64 features

    self.decoder = tf.keras.Sequential([
       layers.Conv2DTranspose(8, kernel_size=3, strides=2, activation='relu', padding='same'), # 16x16x8
       layers.Conv2DTranspose(16, kernel_size=3, strides=2, activation='relu', padding='same'), # 32x32x16
        layers.Conv2DTranspose(1, kernel_size=3, strides=2, activation='sigmoid', padding='same')]) # 64x64x1

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded

autoencoder_objetos = AutoencoderObjetos()
autoencoder_objetos.compile(optimizer='adam', loss=losses.MeanSquaredError())

# Reshape x_train and x_test to include the channel dimension
x_train_reshaped = np.expand_dims(x_train, axis=-1)
x_test_reshaped = np.expand_dims(x_test, axis=-1)

history = autoencoder_objetos.fit(x_train_reshaped, x_train_reshaped,
                                  epochs=25,
                                  batch_size=16,
                                  validation_data=(x_test_reshaped, x_test_reshaped),
                                  shuffle=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print(f"Dimensiones de las etiquetas de entrenamiento codificadas: {y_train_encoded.shape}")
print(f"Dimensiones de las etiquetas de prueba codificadas: {y_test_encoded.shape}")

## Extraer características con el encoder
Utilizar la parte del encoder del autoencoder entrenado para extraer las representaciones latentes (features) de las imágenes de entrenamiento y prueba.


## Dimensionalidad mas baja (64)

In [ ]:
encoded_train_features = autoencoder_objetos.encoder(x_train_reshaped).numpy()
encoded_test_features = autoencoder_objetos.encoder(x_test_reshaped).numpy()

print(f"Dimensiones de las características de entrenamiento codificadas: {encoded_train_features.shape}")
print(f"Dimensiones de las características de prueba codificadas: {encoded_test_features.shape}")

## Entrenamiento y evaluacion de CNN con dimension mas baja

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

# Determine the number of unique classes
num_classes = len(np.unique(y_train_encoded))
print(f"Number of classes: {num_classes}")

# Define the CNN model
classifier_model = Sequential([
    Conv2D(16, (3, 3), activation='relu', input_shape=encoded_train_features.shape[1:]),
    # Removed MaxPooling2D here as it was causing the dimension issue
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)), # Added MaxPooling2D after the second Conv2D
    Flatten(),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

# Compile the CNN model
classifier_model.compile(optimizer=Adam(learning_rate=0.002),
                         loss=SparseCategoricalCrossentropy(),
                         metrics=['accuracy'])

# Train the CNN model
history_classifier = classifier_model.fit(encoded_train_features, y_train_encoded,
                                          epochs=25, # Reduced epochs for quicker demonstration
                                          batch_size=16,
                                          validation_data=(encoded_test_features, y_test_encoded))

### Accuracy con diemsion 64

In [ ]:
loss, accuracy = classifier_model.evaluate(encoded_test_features, y_test_encoded)

print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

## Realizar predicciones en nuevos datos con CNN entrenado

Mostrar cómo usar el clasificador entrenado para predecir el tipo de objeto en nuevas imágenes o fotogramas de video.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select a few images from the test set for prediction
num_images_to_predict = 5
selected_indices = np.random.choice(len(x_test_reshaped), num_images_to_predict, replace=False)

selected_images = x_test_reshaped[selected_indices]
selected_true_labels_encoded = y_test_encoded[selected_indices]
selected_true_labels = label_encoder.inverse_transform(selected_true_labels_encoded)

# Extract features using the encoder
encoded_selected_images = autoencoder_objetos.encoder(selected_images).numpy()

# Predict classes using the trained classifier
predicted_probs = classifier_model.predict(encoded_selected_images)
predicted_labels_encoded = np.argmax(predicted_probs, axis=1)
predicted_labels = label_encoder.inverse_transform(predicted_labels_encoded)

# Display the results
plt.figure(figsize=(15, 6))
for i in range(num_images_to_predict):
    ax = plt.subplot(1, num_images_to_predict, i + 1)
    plt.imshow(np.squeeze(selected_images[i]), cmap='gray')
    plt.title(f"True: {selected_true_labels[i]}\nPred: {predicted_labels[i]}")
    plt.axis('off')
plt.show()

In [ ]:
# Get the dimension of the latent space from the encoder output shape
# The shape is (batch_size, height, width, channels)
# The dimension of the latent space is height * width * channels
latent_space_dimension = encoded_train_features.shape[1] * encoded_train_features.shape[2] * encoded_train_features.shape[3]

print(f"Dimensión del espacio de latencia: {latent_space_dimension}")

# Get the test accuracy from the evaluation of the classifier model
# The accuracy was already calculated and stored in the 'accuracy' variable

# Calculate the ratio of latent space dimension to accuracy
ratio_latent_space_accuracy = latent_space_dimension / accuracy

print(f"Relación Espacio de Latencia / Accuracy: {ratio_latent_space_accuracy}")

In [ ]:
# Exportar el modelo Autoencoder a formato .h5
autoencoder_objetos.save('autoencoder_objetos.h5')
print("Modelo Autoencoder exportado a 'autoencoder_objetos.h5'")

# Exportar el modelo clasificador a formato .h5
classifier_model.save('classifier_model.h5')
print("Modelo clasificador exportado a 'classifier_model.h5'")